In [54]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer


In [38]:
# train_df = pd.read_csv('../artifacts/train.csv')
# test_df = pd.read_csv('../artifacts/test.csv')
df = pd.read_csv('../artifacts/raw.csv')

C:\Users\Dhvanish\AppData\Local\Temp\ipykernel_15108\13393110.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../artifacts/raw.csv')


In [39]:
df.isnull().sum()

Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64

## Imputing missing values ##

In [40]:
df['CompetitionDistance'].fillna(0,inplace=True)
#train_df.isnull().sum()

In [41]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

df['CompetitionOpen_missing'] = df['CompetitionOpenSinceYear'].isna().astype(int)
df['CompetitionOpenSinceMonth'].fillna(1,inplace=True)
df['CompetitionOpenSinceYear'].fillna(df['Year'].min(),inplace=True)
df['CompetitionOpen'] = 12 * (df['Year'] - df['CompetitionOpenSinceYear']) + (df['Month'] - df['CompetitionOpenSinceMonth'])
df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))


In [45]:
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Promo2OpenSinceMonths'] = 12 * (df['Year'] - df['Promo2SinceYear']) + (df['WeekOfYear'] - df['Promo2SinceWeek']) / 4.0
df['Promo2OpenSinceMonths'] = df['Promo2OpenSinceMonths'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)
df.loc[df['Promo2'] == 0, 'Promo2OpenSinceMonths'] = 0
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthStr'] = df['Month'].map(month_map)
promo_months = df['PromoInterval'].fillna('').str.split(',')
df['IsPromoMonth'] = [
    1 if m in months else 0 
    for m, months in zip(df['MonthStr'], promo_months)
]


In [46]:
df.drop(['Store','Date','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)

In [43]:
df.sample(10)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen
865418,964,5,2013-05-17,9319,1359,1,1,0,0,a,...,2013.0,1,5.0,2013.0,"Feb,May,Aug,Nov",2013,5,17,1,4.0
980295,996,7,2013-02-03,0,0,0,0,0,0,c,...,2015.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",2013,2,3,0,0.0
60307,98,7,2015-06-07,0,0,0,0,0,0,d,...,2006.0,1,1.0,2012.0,"Jan,Apr,Jul,Oct",2015,6,7,0,102.0
822413,329,1,2013-06-24,5671,585,1,0,0,0,a,...,1990.0,1,22.0,2012.0,"Mar,Jun,Sept,Dec",2013,6,24,0,276.0
18048,209,3,2015-07-15,7470,732,1,1,0,0,a,...,2011.0,1,31.0,2013.0,"Jan,Apr,Jul,Oct",2015,7,15,0,46.0
766981,647,2,2013-08-13,7272,822,1,1,0,0,a,...,2013.0,0,NaN,NaN,NaN,2013,8,13,0,4.0
77895,961,6,2015-05-23,5874,623,1,0,0,0,d,...,2013.0,0,NaN,NaN,NaN,2015,5,23,1,28.0
508378,724,3,2014-04-02,6685,666,1,1,0,0,d,...,2013.0,0,NaN,NaN,NaN,2014,4,2,1,15.0
81652,258,2,2015-05-19,7458,530,1,1,0,0,a,...,2010.0,1,37.0,2009.0,"Jan,Apr,Jul,Oct",2015,5,19,0,58.0
341974,1044,3,2014-09-10,4942,734,1,0,0,1,c,...,2015.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",2014,9,10,0,0.0


In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 19 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   DayOfWeek                1017209 non-null  int64  
 1   Sales                    1017209 non-null  int64  
 2   Customers                1017209 non-null  int64  
 3   Open                     1017209 non-null  int64  
 4   Promo                    1017209 non-null  int64  
 5   StateHoliday             1017209 non-null  object 
 6   SchoolHoliday            1017209 non-null  int64  
 7   StoreType                1017209 non-null  object 
 8   Assortment               1017209 non-null  object 
 9   CompetitionDistance      1017209 non-null  float64
 10  Promo2                   1017209 non-null  int64  
 11  Year                     1017209 non-null  int32  
 12  Month                    1017209 non-null  int32  
 13  Day                      1017209 non-null 

In [47]:
num_col=['Customers','Promo','CompetitionDistance','CompetitionOpen','WeekOfYear','Promo2OpenSinceMonths','Day']
cat_col = ['DayOfWeek','StateHoliday','StoreType','Assortment','Year','Month']

In [48]:
X = df.drop(columns=['Sales'])
y = df['Sales']

In [49]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [ ]:
preprocessor = ColumnTransformer([
    ('scl',StandardScaler(),num_col),
    ('ohe',OneHotEncoder(drop='first'),cat_col)    
])

In [ ]:
preprocessor.fit_transform(X_train)
preprocessor.transform(X_test)

TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['int', 'str']